# Semantic Similarity Analysis — AI Actors Network

**Objectif** : Analyser les similarités sémantiques entre entreprises de l'IA à partir de leurs descriptions textuelles.

**Prompt utilisateur** :  
_"On va faire une analyse des similarités sémantiques des entreprises de notre base de données sur un modèle analogue à l'analyse de compétition._  
_1) on sélectionne les entreprises qui ont une valorisation supérieure à 100, et dont le champs description n'est pas vide._  
_2) on calcule leurs embeddings avec un modèle multilingue pour tenir compte de l'hétérogénéité des langues (il y a du français aussi)_  
_3) tu calculeras la similarité cos tu la transforme en distance_  
_4) tu appliques un mds avec les mêmes caractéristiques pour l'analyse de compétition._  
_Enfin tu prendras soin de documenter le MD du carnet jupyter et de chacune de ses sections et étapes avec mes prompts"_

## Pipeline

1. **Extraction SQL** : entreprises avec capitalisation/fonds > 100M et description non vide
2. **Embeddings multilingues** : sentence-transformers (modèle paraphrase-multilingual)
3. **Matrice de similarité** : cosine → distance euclidienne
4. **Projection MDS 2D** : réduction dimensionnelle pour visualisation
5. **Export et visualisation** : carte interactive Plotly

## 1. Configuration et imports

**Prompt utilisateur** : _"on calcule leurs embeddings avec un modèle multilingue"_

Installation des dépendances nécessaires :
```bash
pip install sentence-transformers scikit-learn pandas plotly numpy
```

In [ ]:
import sqlite3
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
from sentence_transformers import SentenceTransformer
from sklearn.manifold import MDS
from sklearn.metrics.pairwise import cosine_similarity

ROOT        = Path(__file__).parent.parent if "__file__" in dir() else Path().resolve().parent
DB_PATH     = ROOT / "database.db"
EXPORTS_DIR = ROOT / "analyses" / "exports"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

print(f"Base : {DB_PATH}")
print(f"Exports : {EXPORTS_DIR}")

## 2. Extraction SQL — entreprises avec valorisation > 100M et description

**Prompt utilisateur** : _"on sélectionne les entreprises qui ont une valorisation supérieure à 100, et dont le champs description n'est pas vide"_

Critères de sélection :
- `MAX(capitalization, funds_raised) > 100` (en millions)
- `description IS NOT NULL AND description != ''`
- Tri par valorisation décroissante

In [ ]:
SQL = """
SELECT
    id,
    name,
    description,
    sector,
    country,
    founded_year,
    MAX(
        CAST(REPLACE(IFNULL(capitalization, '0'), ',', '.') AS REAL),
        CAST(REPLACE(IFNULL(funds_raised,    '0'), ',', '.') AS REAL)
    ) AS valuation
FROM enterprises
WHERE description IS NOT NULL
  AND description != ''
  AND MAX(
        CAST(REPLACE(IFNULL(capitalization, '0'), ',', '.') AS REAL),
        CAST(REPLACE(IFNULL(funds_raised,    '0'), ',', '.') AS REAL)
      ) > 100
ORDER BY valuation DESC
"""

with sqlite3.connect(DB_PATH) as con:
    df_raw = pd.read_sql_query(SQL, con)

print(f"{len(df_raw)} entreprises sélectionnées")
print(f"Valorisation min : {df_raw['valuation'].min():.0f}M | max : {df_raw['valuation'].max():.0f}M")
df_raw[["name", "valuation", "description"]].head(10)

In [ ]:
# Export des données brutes
raw_path = EXPORTS_DIR / "semantic_raw.csv"
df_raw.to_csv(raw_path, index=False)
print(f"Export brut → {raw_path}")

## 3. Génération des embeddings multilingues

**Prompt utilisateur** : _"on calcule leurs embeddings avec un modèle multilingue pour tenir compte de l'hétérogénéité des langues (il y a du français aussi)"_

Modèle utilisé : **`paraphrase-multilingual-MiniLM-L12-v2`**
- Support de 50+ langues (français, anglais, allemand, etc.)
- Dimension : 384
- Optimisé pour les tâches de similarité sémantique

⚠️ **Attention** : le téléchargement du modèle peut prendre quelques minutes au premier lancement.

In [ ]:
# Chargement du modèle multilingue
print("Chargement du modèle sentence-transformers...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"Modèle chargé : {model.get_sentence_embedding_dimension()} dimensions")

# Génération des embeddings
descriptions = df_raw['description'].tolist()
print(f"Génération des embeddings pour {len(descriptions)} descriptions...")
embeddings = model.encode(descriptions, show_progress_bar=True, batch_size=32)

print(f"Embeddings shape : {embeddings.shape}")
print(f"Exemple (première entreprise) : {embeddings[0][:5]}... (5 premières dimensions)")

## 4. Calcul de similarité cosine et conversion en distance

**Prompt utilisateur** : _"tu calculeras la similarité cos tu la transforme en distance"_

- **Similarité cosine** : $\text{sim}(A, B) = \frac{A \cdot B}{||A|| \cdot ||B||}$ ∈ [-1, 1]
- **Conversion en distance** : $\text{dist}(A, B) = 1 - \text{sim}(A, B)$ ∈ [0, 2]

La diagonale (auto-similarité) est mise à 0.

In [ ]:
# Calcul de la similarité cosine
similarity_matrix = cosine_similarity(embeddings)
print(f"Matrice de similarité : {similarity_matrix.shape}")
print(f"Similarité min : {similarity_matrix.min():.3f} | max : {similarity_matrix.max():.3f}")

# Conversion en distance
distance_matrix = 1 - similarity_matrix
np.fill_diagonal(distance_matrix, 0)

print(f"Matrice de distance : {distance_matrix.shape}")
print(f"Distance min (hors diagonale) : {distance_matrix[distance_matrix > 0].min():.3f}")
print(f"Distance max : {distance_matrix.max():.3f}")

# Export de la matrice de distance
df_distance = pd.DataFrame(
    distance_matrix,
    index=df_raw['name'],
    columns=df_raw['name']
)
distance_path = EXPORTS_DIR / "semantic_distance_matrix.csv"
df_distance.to_csv(distance_path)
print(f"Export matrice de distance → {distance_path}")

## 5. Projection MDS 2D

**Prompt utilisateur** : _"tu appliques un mds avec les mêmes caractéristiques pour l'analyse de compétition"_

Configuration MDS :
- **Dissimilarité** : matrice de distance pré-calculée
- **Dimensions** : 2 (visualisation 2D)
- **Initialisation** : aléatoire (reproductible avec seed)
- **Normalized stress** : auto (recommandation scikit-learn)

Le MDS préserve au mieux les distances entre entreprises dans l'espace réduit.

In [ ]:
N = len(df_raw)
print(f"Projection MDS pour {N} entreprises...")

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        init="random",
        random_state=RANDOM_SEED,
        normalized_stress="auto"
    )
    coords = mds.fit_transform(distance_matrix)

print(f"MDS stress : {mds.stress_:.2f}")
print(f"Coordonnées shape : {coords.shape}")

# Construction du DataFrame de coordonnées
df_coords = pd.DataFrame({
    "name": df_raw['name'],
    "x": coords[:, 0],
    "y": coords[:, 1],
    "valuation": df_raw['valuation'],
    "log_valuation": np.log10(df_raw['valuation'] + 1),
    "sector": df_raw['sector'].fillna("Unknown"),
    "country": df_raw['country'].fillna("Unknown"),
    "description": df_raw['description']
})

# Extraction du secteur principal
df_coords["sector"] = df_coords["sector"].apply(
    lambda s: s.split(",")[0].strip() if pd.notna(s) and s != "Unknown" else "Unknown"
)

coords_path = EXPORTS_DIR / "semantic_coords_2d.csv"
df_coords.to_csv(coords_path, index=False)
print(f"Coordonnées 2D → {coords_path}")

df_coords.sort_values("valuation", ascending=False).head(10)

## 6. Visualisation interactive Plotly

**Prompt utilisateur** : _"tu appliques un mds avec les mêmes caractéristiques pour l'analyse de compétition"_

Carte interactive avec :
- **Couleur** : secteur d'activité
- **Taille** : valorisation (logarithmique)
- **Hover** : nom, secteur, pays, valorisation, description
- **Style** : cohérent avec l'analyse de compétition (fond #FDFAF4)

In [ ]:
# Préparation du hover text
def fmt_hover(row):
    lines = [f"<b>{row['name']}</b>"]
    if pd.notna(row.get("sector")) and row["sector"] != "Unknown":
        lines.append(f"Sector: {row['sector']}")
    if pd.notna(row.get("country")) and row["country"] != "Unknown":
        lines.append(f"Country: {row['country']}")
    val = float(row.get("valuation") or 0)
    if val > 1000:
        lines.append(f"Valuation: {val/1000:.1f}B USD")
    else:
        lines.append(f"Valuation: {val:.0f}M USD")
    if pd.notna(row.get("description")):
        desc = str(row["description"])
        snippet = desc[:200].rstrip()
        lines.append(f"<i>{snippet}{'…' if len(desc) > 200 else ''}</i>")
    return "<br>".join(lines)

df_coords["hover"] = df_coords.apply(fmt_hover, axis=1)
df_coords["marker_size"] = np.maximum(df_coords["log_valuation"] * 3, 5)

fig = px.scatter(
    df_coords,
    x="x", y="y",
    color="sector",
    size="marker_size",
    size_max=25,
    text="name",
    custom_data=["hover"],
    color_discrete_sequence=px.colors.qualitative.Light24,
    title=f"Similarité sémantique — MDS ({N} entreprises · taille ∝ log valorisation)",
    labels={"x": "Dimension 1", "y": "Dimension 2", "sector": "Secteur"},
    width=1400,
    height=1200,
)

fig.update_traces(
    hovertemplate="%{customdata[0]}<extra></extra>",
    textposition="top center",
    textfont=dict(size=7),
    marker=dict(opacity=0.78, line=dict(width=0.4, color="white")),
)

fig.update_layout(
    legend=dict(title="Secteur", font=dict(size=10)),
    font=dict(family="Inter, sans-serif", size=11),
    plot_bgcolor="#FDFAF4",
    paper_bgcolor="#FDFAF4",
    hovermode="closest",
)

fig_html = EXPORTS_DIR / "semantic_similarity_map_2d.html"
fig.write_html(str(fig_html))
print(f"Carte interactive → {fig_html}")
fig.show()

## 7. Analyse des clusters sémantiques

Identification des entreprises les plus similaires par paires.

In [ ]:
# Top 20 paires les plus similaires
n = len(df_raw)
pairs = []

for i in range(n):
    for j in range(i+1, n):
        pairs.append({
            "company_1": df_raw.iloc[i]['name'],
            "company_2": df_raw.iloc[j]['name'],
            "similarity": similarity_matrix[i, j],
            "distance": distance_matrix[i, j]
        })

df_pairs = pd.DataFrame(pairs).sort_values("similarity", ascending=False)

print("Top 20 paires les plus similaires :")
df_pairs.head(20)

In [ ]:
# Export des paires
pairs_path = EXPORTS_DIR / "semantic_similarity_pairs.csv"
df_pairs.to_csv(pairs_path, index=False)
print(f"Export paires de similarité → {pairs_path}")

## 8. Récapitulatif des exports

In [ ]:
print("── Récapitulatif des exports ────────────────────────────")
for p in [raw_path, distance_path, coords_path, pairs_path, fig_html]:
    size_kb = Path(p).stat().st_size / 1024
    print(f"  {p.name:<42} {size_kb:6.1f} KB")